In [ ]:
import sys, torch
print(sys.executable)                  # should contain envs\GPU
print(torch.__version__)               # should be 2.4.1+cu121
print(torch.cuda.is_available())       # should be True
print(torch.cuda.get_device_name(0))

In [ ]:
 !pip install pandas pyarrow tqdm

In [ ]:
import os, glob
import pyarrow.parquet as pq
import pyarrow as pa
import pandas as pd
from tqdm import tqdm

SRC = r"G:\Stanford_Chexpert"      # where your parquet files are
OUT = r"G:\chexpert_images"        # where the JPGs will go
FRONTAL_ONLY = True                # skip lateral images to save space

files = sorted(glob.glob(os.path.join(SRC, "**", "*.parquet"), recursive=True))
print(len(files), "parquet files found")     # expect 24

# find the image column automatically (the column that holds {bytes, path})
schema = pq.read_schema(files[0])
IMG_COL = [f.name for f in schema
           if pa.types.is_struct(f.type) and "bytes" in [x.name for x in f.type]][0]
print("image column:", IMG_COL)
print("all columns:", schema.names)

os.makedirs(OUT, exist_ok=True)
tables = []

for f in tqdm(files, desc="files"):
    pf = pq.ParquetFile(f)
    for batch in pf.iter_batches(batch_size=500):
        df = batch.to_pandas()
        if FRONTAL_ONLY:
            df = df[df["Path"].str.contains("frontal")]
        for _, row in df.iterrows():
            # "CheXpert-v1.0-small/train/patient00001/study1/view1_frontal.jpg"
            rel = row["Path"].replace("CheXpert-v1.0-small/", "")
            dest = os.path.join(OUT, *rel.split("/"))
            os.makedirs(os.path.dirname(dest), exist_ok=True)
            if not os.path.exists(dest):
                with open(dest, "wb") as fh:
                    fh.write(row[IMG_COL]["bytes"])
        tables.append(df.drop(columns=[IMG_COL]))

meta = pd.concat(tables, ignore_index=True)
meta["patient_id"] = meta["Path"].str.extract(r"(patient\d+)")
meta["img_path"] = meta["Path"].str.replace("CheXpert-v1.0-small/", "", regex=False) \
                       .apply(lambda p: os.path.join(OUT, *p.split("/")))
meta.to_csv(os.path.join(OUT, "chexpert_all.csv"), index=False)

print(len(meta), "images saved,", meta["patient_id"].nunique(), "patients")

In [ ]:
import pandas as pd
meta = pd.read_csv(r"G:\chexpert_images\chexpert_all.csv")
print(len(meta), meta["patient_id"].nunique())

In [ ]:
import json
info = json.loads(pq.read_schema(files[0]).metadata[b"huggingface"])["info"]["features"]
for k, v in info.items():
    if v.get("_type") == "ClassLabel":
        print(k, v["names"])

In [ ]:
import pandas as pd, numpy as np

meta = pd.read_csv(r"G:\chexpert_images\chexpert_all.csv")

print("rows:", len(meta))
print("patients:", meta["patient_id"].nunique())
print()
print("Sex values:", meta["Sex"].unique())
print("Cardiomegaly values:", sorted(meta["Cardiomegaly"].unique()))
print()
print(meta[["Path", "patient_id", "Sex", "Age", "Cardiomegaly"]].head())

In [ ]:
LABELS = ["Cardiomegaly", "Edema", "Consolidation", "Atelectasis", "Pneumothorax", "Pleural Effusion"]
PRESENT = 3   # 0 unlabeled, 1 uncertain, 2 absent, 3 present

for l in LABELS + ["Support Devices"]:
    meta[l + "_raw"] = meta[l]                      # keep the original number
    meta[l] = (meta[l] == PRESENT).astype(int)      # only "present" becomes 1, everything else 0

meta["Sex"] = meta["Sex"].map({0: "Male", 1: "Female"})
meta["origin"] = meta["Path"].str.extract(r"/(train|valid)/")[0]

print(meta["origin"].value_counts())
print()
print("How common is each disease (fraction of images):")
print(meta[LABELS].mean().round(3))
print()
print(meta[["Cardiomegaly_raw", "Cardiomegaly"]].head(10))

In [ ]:
import pandas as pd

# 1. Load the original file (always start from the untouched CSV)
meta = pd.read_csv(r"G:\chexpert_images\chexpert_all.csv")

# 2. Clean the labels
LABELS = ["Cardiomegaly", "Edema", "Consolidation", "Atelectasis", "Pneumothorax", "Pleural Effusion"]
PRESENT = 3   # 0 unlabeled, 1 uncertain, 2 absent, 3 present

for l in LABELS + ["Support Devices"]:
    meta[l + "_raw"] = meta[l]                      # keep the original number
    meta[l] = (meta[l] == PRESENT).astype(int)      # only "present" = 1, everything else = 0

meta["Sex"] = meta["Sex"].map({0: "Male", 1: "Female"})
meta["origin"] = meta["Path"].str.extract(r"/(train|valid)/")[0]

# 3. Save
meta.to_csv(r"G:\chexpert_cleaned_labels.csv", index=False)

# 4. Check
print("Saved: G:\\chexpert_cleaned_labels.csv")
print(len(meta), "rows")
print(meta["origin"].value_counts())
print()
print("Fraction of images with each disease:")
print(meta[LABELS].mean().round(3))

In [ ]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# 1. Load the cleaned file from the last step
meta = pd.read_csv(r"G:\chexpert_cleaned_labels.csv")
LABELS = ["Cardiomegaly", "Edema", "Consolidation", "Atelectasis", "Pneumothorax", "Pleural Effusion"]

# 2. Separate the radiologist-labeled set
train_all = meta[meta.origin == "train"].copy()
valid_expert = meta[meta.origin == "valid"].copy()

# 3. Split train_all by patient: 80% train, 10% val, 10% test
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr, tmp = next(gss.split(train_all, groups=train_all["patient_id"]))
tmp_df = train_all.iloc[tmp]
va, te = next(GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=42)
              .split(tmp_df, groups=tmp_df["patient_id"]))

train_all["split"] = "train"
train_all.loc[tmp_df.index[va], "split"] = "val"
train_all.loc[tmp_df.index[te], "split"] = "test"
valid_expert["split"] = "expert_valid"

# 4. Safety check: no patient in two splits
assert (train_all.groupby("patient_id")["split"].nunique() == 1).all(), "Leakage!"

# 5. Save the final file
final = pd.concat([train_all, valid_expert], ignore_index=True)
final.to_csv(r"G:\chexpert_final.csv", index=False)

print(final["split"].value_counts())
print()
print("Patients per split:")
print(final.groupby("split")["patient_id"].nunique())
print("\nSaved: G:\\chexpert_final.csv")

In [ ]:
import sys; sys.path.append("G:/")
import os, time, numpy as np, pandas as pd, torch, timm, torch.nn as nn
import torchvision.transforms as T
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from sklearn.metrics import (roc_auc_score, roc_curve, f1_score, precision_score,
                             recall_score, multilabel_confusion_matrix)
from tqdm import tqdm
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from chex_ds import DS, LABELS

MODEL, SEED, EPOCHS, BS, LR, SIZE = "densenet121", 0, 7, 64, 1e-4, 224
SUBSET  = 0          # 0 = all data
WORKERS = 4          # set to 0 if it hangs
OUT = r"G:\runs"
FIG = os.path.join(OUT, "figs")
os.makedirs(FIG, exist_ok=True)
tag = f"{MODEL}_seed{SEED}"
FIG = os.path.join(OUT, "figs", tag) 
torch.manual_seed(SEED); np.random.seed(SEED)
torch.backends.cudnn.benchmark = True

# ---- IEEE figure style: Times, 8 pt, vector PDF + 600 dpi PNG ----
plt.rcParams.update({
    "font.family": "serif", "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix", "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7,
    "axes.linewidth": 0.6, "lines.linewidth": 1.0, "grid.linewidth": 0.4,
    "figure.dpi": 150, "pdf.fonttype": 42, "ps.fonttype": 42,
})
COL1, COL2 = 3.5, 7.16      # IEEE single / double column width (inches)

def save_fig(fig, name):
    fig.savefig(os.path.join(FIG, name + ".pdf"), bbox_inches="tight")
    fig.savefig(os.path.join(FIG, name + ".png"), dpi=600, bbox_inches="tight")

print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
CLEAN = r"G:\chexpert_final_clean.csv"

if os.path.exists(CLEAN):
    df = pd.read_csv(CLEAN)
else:
    df = pd.read_csv(r"G:\chexpert_final.csv")

    def is_ok(p):
        try:
            with Image.open(p) as im:
                im.load()
            return True
        except Exception:
            return False

    with ThreadPoolExecutor(16) as pool:
        ok = np.array(list(tqdm(pool.map(is_ok, df.img_path), total=len(df), desc="check images")))
    print(f"dropped {(~ok).sum()} of {len(df)} bad images")
    print(df.img_path[~ok].tolist()[:10])
    df = df[ok].reset_index(drop=True)
    df.to_csv(CLEAN, index=False)

if SUBSET:
    df = df.groupby("split").head(SUBSET)
print(df.split.value_counts())

In [ ]:
feat = df[LABELS + ["Support Devices"]].copy()
feat["Age"] = df["Age"]
feat["Sex (F=1)"] = (df["Sex"] == "Female").astype(int)
corr = feat.corr(method="pearson")
names, C = corr.columns.tolist(), corr.values

fig, ax = plt.subplots(figsize=(COL2 * 0.75, COL2 * 0.7), constrained_layout=True)
im = ax.imshow(np.ma.array(C, mask=np.triu(np.ones_like(C, dtype=bool), k=1)),
               cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=45, ha="right", rotation_mode="anchor")
ax.set_yticks(range(len(names))); ax.set_yticklabels(names)
for i in range(len(names)):
    for j in range(i + 1):
        ax.text(j, i, f"{C[i, j]:.2f}", ha="center", va="center", fontsize=6,
                color="white" if abs(C[i, j]) > 0.6 else "black")
ax.grid(False)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03).set_label("Pearson $r$")
save_fig(fig, "fig_feature_correlation"); plt.show()

In [ ]:
WORKERS = 4      # if anything hangs, set to 2

norm = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
tf_tr = T.Compose([T.Resize((SIZE, SIZE)), T.RandomHorizontalFlip(), T.ToTensor(), norm])
tf_ev = T.Compose([T.Resize((SIZE, SIZE)), T.ToTensor(), norm])

def mk(split, tf, shuffle):
    d = df[df.split == split].reset_index(drop=True)
    return d, DataLoader(DS(d, tf), batch_size=BS, shuffle=shuffle, num_workers=WORKERS,
                         pin_memory=False, persistent_workers=False,
                         timeout=300 if WORKERS > 0 else 0)

tr_df, tr = mk("train", tf_tr, True)          # augmented, used for training
_,     tr_eval = mk("train", tf_ev, False)    # same images, no augmentation, for final metrics
va_df, va = mk("val", tf_ev, False)
te_df, te = mk("test", tf_ev, False)
ex_df, ex = mk("expert_valid", tf_ev, False)

print("images train/val/test/expert:", len(tr_df), len(va_df), len(te_df), len(ex_df))

In [ ]:
model = timm.create_model(MODEL, pretrained=True, num_classes=len(LABELS)).cuda()
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS * len(tr))
lossf, scaler = nn.BCEWithLogitsLoss(), torch.amp.GradScaler("cuda")

@torch.no_grad()
def predict(model, dl):
    model.eval(); P, Y = [], []
    for x, y in dl:
        with torch.autocast("cuda"):
            out = model(x.cuda(non_blocking=True))
        P.append(torch.sigmoid(out.float()).cpu()); Y.append(y)
    return torch.cat(P).numpy(), torch.cat(Y).numpy()

def aucs(P, Y):
    out = []
    for i in range(len(LABELS)):
        try: out.append(roc_auc_score(Y[:, i], P[:, i]))
        except ValueError: out.append(float("nan"))
    return out

def bce(P, Y, eps=1e-7):
    P = np.clip(P, eps, 1 - eps)
    return float(-(Y * np.log(P) + (1 - Y) * np.log(1 - P)).mean())

def evaluate(P, Y, thr=0.5):
    pred = (P >= np.broadcast_to(thr, (Y.shape[1],))).astype(int)
    Yi = Y.astype(int)
    f1s = [f1_score(Yi[:, i], pred[:, i], zero_division=0) for i in range(Yi.shape[1])]
    return dict(loss=bce(P, Y), acc=float((pred == Yi).mean()),
                exact=float((pred == Yi).all(1).mean()),
                f1_macro=float(np.mean(f1s)),
                f1_micro=float(f1_score(Yi, pred, average="micro", zero_division=0)),
                auroc=float(np.nanmean(aucs(P, Y))))
print("model ready:", MODEL)

In [ ]:
BEST = rf"{OUT}\{tag}.pt"
CKPT = rf"{OUT}\{tag}_last.pt"
HIST = rf"{OUT}\{tag}_history.csv"
RESUME = True
hist, best, start = [], 0.0, 0

if RESUME and os.path.exists(CKPT):
    ck = torch.load(CKPT, map_location="cuda", weights_only=False)
    model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
    sched.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"])
    hist, best, start = ck["hist"], ck["best"], ck["ep"] + 1
    print("resumed, next epoch:", start + 1)

for ep in range(start, EPOCHS):
    model.train(); t0 = time.time()
    run_loss = torch.zeros((), device="cuda"); run_acc = torch.zeros((), device="cuda"); n = 0
    for x, y in tqdm(tr, desc=f"epoch {ep+1}/{EPOCHS}"):
        x, y = x.cuda(non_blocking=True), y.cuda(non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.autocast("cuda"):
            out = model(x); loss = lossf(out, y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        b = x.size(0)
        run_loss += loss.detach() * b
        run_acc  += ((out.detach() > 0) == (y > 0.5)).float().mean() * b
        n += b

    P, Y = predict(model, va); v = evaluate(P, Y)
    row = dict(epoch=ep + 1, train_loss=float(run_loss) / n, train_acc=float(run_acc) / n,
               val_loss=v["loss"], val_acc=v["acc"], val_f1=v["f1_macro"], val_auroc=v["auroc"],
               sec=time.time() - t0)
    hist.append(row); pd.DataFrame(hist).to_csv(HIST, index=False)
    print(f"ep {ep+1:2d} | loss {row['train_loss']:.4f}/{row['val_loss']:.4f} | "
          f"acc {row['train_acc']:.4f}/{row['val_acc']:.4f} | val F1 {row['val_f1']:.4f} | "
          f"val AUROC {row['val_auroc']:.4f} | {row['sec']/60:.1f} min")

    if v["auroc"] > best:
        best = v["auroc"]; torch.save(model.state_dict(), BEST)
    torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
                    scaler=scaler.state_dict(), hist=hist, best=best, ep=ep), CKPT)
print("done, best val AUROC:", round(best, 4))

In [ ]:
import shutil
BK = rf"{OUT}\backup_e15_seed0"
os.makedirs(BK, exist_ok=True)
shutil.copytree(FIG, rf"{BK}\figs", dirs_exist_ok=True)
for f in os.listdir(OUT):
    if f.endswith((".tex", ".csv", ".pt")) and os.path.isfile(rf"{OUT}\{f}"):
        shutil.copy(rf"{OUT}\{f}", BK)
np.save(rf"{BK}\thresholds.npy", THR)
print("backed up to", BK)

In [ ]:
h = pd.read_csv(HIST)
fig, ax = plt.subplots(1, 3, figsize=(COL2, 2.2))
ax[0].plot(h.epoch, h.train_loss, "-o", ms=2.5, label="Train")
ax[0].plot(h.epoch, h.val_loss, "--s", ms=2.5, label="Validation")
ax[0].set(xlabel="Epoch", ylabel="BCE loss")
ax[1].plot(h.epoch, h.train_acc, "-o", ms=2.5, label="Train")
ax[1].plot(h.epoch, h.val_acc, "--s", ms=2.5, label="Validation")
ax[1].set(xlabel="Epoch", ylabel="Accuracy")
ax[2].plot(h.epoch, h.val_f1, "-o", ms=2.5, label="Macro-F1")
ax[2].plot(h.epoch, h.val_auroc, "--s", ms=2.5, label="Mean AUROC")
ax[2].set(xlabel="Epoch", ylabel="Validation score")
for a in ax: a.grid(True, alpha=0.3); a.legend(frameon=False)
fig.tight_layout(); save_fig(fig, "fig_training_curves"); plt.show()

In [ ]:
@torch.no_grad()
def predict(model, dl, desc="eval"):
    model.eval(); P, Y = [], []
    for x, y in tqdm(dl, desc=desc):
        with torch.autocast("cuda"):
            out = model(x.cuda(non_blocking=True))
        P.append(torch.sigmoid(out.float()).cpu()); Y.append(y)
    return torch.cat(P).numpy(), torch.cat(Y).numpy()

model.load_state_dict(torch.load(BEST, map_location="cuda", weights_only=True))
splits = {"Train": tr_eval, "Validation": va, "Test": te, "Expert valid.": ex}
preds = {}
for k, dl in splits.items():
    preds[k] = predict(model, dl, desc=f"predict {k}")

# per-label threshold chosen on the VALIDATION set only (max F1), then applied everywhere
def best_thr(p, y):
    grid = np.linspace(0.05, 0.95, 91)
    return grid[int(np.argmax([f1_score(y, p >= t, zero_division=0) for t in grid]))]

Pv, Yv = preds["Validation"]
THR = np.array([best_thr(Pv[:, i], Yv[:, i].astype(int)) for i in range(len(LABELS))])
print("thresholds:", dict(zip(LABELS, THR.round(2))))

# overall table
rows = []
for k, (P, Y) in preds.items():
    m = evaluate(P, Y, THR)
    rows.append({"Split": k, "N": len(Y), "Accuracy": m["acc"], "Exact match": m["exact"],
                 "Macro-F1": m["f1_macro"], "Micro-F1": m["f1_micro"], "AUROC": m["auroc"]})
summary = pd.DataFrame(rows)
print(summary.round(4).to_string(index=False))

# per-label table
per = []
for k, (P, Y) in preds.items():
    pred, Yi = (P >= THR).astype(int), Y.astype(int)
    for i, l in enumerate(LABELS):
        try: auc = roc_auc_score(Yi[:, i], P[:, i])
        except ValueError: auc = np.nan
        per.append({"Split": k, "Label": l, "Threshold": THR[i], "Prevalence": Yi[:, i].mean(),
                    "Accuracy": (pred[:, i] == Yi[:, i]).mean(),
                    "Precision": precision_score(Yi[:, i], pred[:, i], zero_division=0),
                    "Recall": recall_score(Yi[:, i], pred[:, i], zero_division=0),
                    "F1": f1_score(Yi[:, i], pred[:, i], zero_division=0), "AUROC": auc})
per = pd.DataFrame(per)
print(per[per.Split == "Test"].drop(columns="Split").round(4).to_string(index=False))

# save CSV + LaTeX (IEEE tables); if to_latex errors: pip install jinja2
summary.to_csv(rf"{OUT}\table_summary.csv", index=False)
per.to_csv(rf"{OUT}\table_per_label.csv", index=False)
open(rf"{OUT}\table_summary.tex", "w").write(summary.to_latex(
    index=False, float_format="%.3f", caption=f"Overall performance of {MODEL} on CheXpert.", label="tab:summary"))
open(rf"{OUT}\table_test_per_label.tex", "w").write(per[per.Split == "Test"].drop(columns=["Split", "Threshold"]).to_latex(
    index=False, float_format="%.3f", caption="Per-label test performance.", label="tab:perlabel"))
np.savez(rf"{OUT}\{tag}_preds.npz", **{k.replace(" ", "_").replace(".", ""): v[0] for k, v in preds.items()})

In [ ]:
P, Y = preds["Test"]
o = te_df[["img_path", "patient_id", "Sex", "Age"]].copy()
for i, l in enumerate(LABELS):
    o["y_" + l] = Y[:, i]; o["p_" + l] = P[:, i]
o.to_csv(rf"{OUT}\{tag}_test_preds.csv", index=False)

groups = {"Male": te_df.Sex.eq("Male"), "Female": te_df.Sex.eq("Female"),
          "Age <50": te_df.Age < 50, "Age 50-65": te_df.Age.between(50, 65),
          "Age >65": te_df.Age > 65}
rows = []
for g, m in groups.items():
    m = m.values
    r = {"Group": g, "N": int(m.sum())}
    for i, l in enumerate(LABELS):
        r[l] = roc_auc_score(Y[m, i], P[m, i])
    r["Mean"] = np.mean([r[l] for l in LABELS])
    rows.append(r)
fair = pd.DataFrame(rows)
fair.to_csv(rf"{OUT}\{tag}_fairness.csv", index=False)
display(fair.round(3))

In [ ]:
rng = np.random.default_rng(0)
pid = te_df.patient_id.values
plist = pd.Series(np.arange(len(te_df))).groupby(pid).apply(np.array).values
sex, age = te_df.Sex.values, te_df.Age.values

def mean_auc(idx):
    return np.mean([roc_auc_score(Y[idx, i], P[idx, i]) for i in range(len(LABELS))])

d_sex, d_age = [], []
for _ in range(500):
    idx = np.concatenate(plist[rng.integers(0, len(plist), len(plist))])
    s, a = sex[idx], age[idx]
    d_sex.append(mean_auc(idx[s == "Female"]) - mean_auc(idx[s == "Male"]))
    d_age.append(mean_auc(idx[a < 50]) - mean_auc(idx[a > 65]))

for name, d in [("Female - Male", d_sex), ("Age<50 - Age>65", d_age)]:
    lo, hi = np.percentile(d, [2.5, 97.5])
    print(f"{name}: {np.mean(d):+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]")

In [ ]:
def plot_cm(P, Y, fname):
    cms = multilabel_confusion_matrix(Y.astype(int), (P >= THR).astype(int))
    fig, axes = plt.subplots(2, 3, figsize=(COL2, 4.3))
    for ax, cm, l in zip(axes.ravel(), cms, LABELS):
        pct = cm / cm.sum(1, keepdims=True).clip(min=1)
        ax.imshow(pct, cmap="Blues", vmin=0, vmax=1)
        for i in range(2):
            for j in range(2):
                ax.text(j, i, f"{cm[i, j]:,}\n({pct[i, j]*100:.1f}%)", ha="center", va="center",
                        fontsize=6.5, color="white" if pct[i, j] > 0.5 else "black")
        ax.set_xticks([0, 1]); ax.set_xticklabels(["Neg", "Pos"])
        ax.set_yticks([0, 1]); ax.set_yticklabels(["Neg", "Pos"])
        ax.set(xlabel="Predicted", ylabel="Actual", title=l); ax.grid(False)
    fig.tight_layout(); save_fig(fig, fname); plt.show()

for k, (P, Y) in preds.items():
    plot_cm(P, Y, "fig_cm_" + k.lower().replace(" ", "_").replace(".", ""))

In [ ]:
P, Y = preds["Test"]
styles = ["-", "--", "-.", ":", (0, (5, 1)), (0, (3, 1, 1, 1))]
fig, ax = plt.subplots(figsize=(COL1, 3.0))
for i, l in enumerate(LABELS):
    fpr, tpr, _ = roc_curve(Y[:, i], P[:, i])
    ax.plot(fpr, tpr, ls=styles[i], label=f"{l} ({roc_auc_score(Y[:, i], P[:, i]):.3f})")
ax.plot([0, 1], [0, 1], "k:", lw=0.6)
ax.set(xlabel="False positive rate", ylabel="True positive rate", xlim=(0, 1), ylim=(0, 1.02))
ax.grid(True, alpha=0.3); ax.legend(loc="lower right", frameon=False)
fig.tight_layout(); save_fig(fig, "fig_roc_test"); plt.show()

In [ ]:
import gc
from sklearn.metrics import multilabel_confusion_matrix

try:
    del model, opt, sched, scaler
except NameError:
    pass
gc.collect(); torch.cuda.empty_cache()

@torch.no_grad()
def _predict(m, dl, desc):
    m.eval(); P, Y = [], []
    for x, y in tqdm(dl, desc=desc, leave=False):
        with torch.autocast("cuda"):
            out = m(x.cuda(non_blocking=True))
        P.append(torch.sigmoid(out.float()).cpu()); Y.append(y)
    return torch.cat(P).numpy(), torch.cat(Y).numpy()

def _best_thr(p, y):
    grid = np.linspace(0.05, 0.95, 91)
    return grid[int(np.argmax([f1_score(y, p >= t, zero_division=0) for t in grid]))]

def run_seed(seed, patience=99, eval_train=True):
    tag = f"{MODEL}_e{EPOCHS}_seed{seed}"
    BEST, CKPT, HIST = rf"{OUT}\{tag}.pt", rf"{OUT}\{tag}_last.pt", rf"{OUT}\{tag}_history.csv"
    D = rf"{OUT}\{tag}_results"; os.makedirs(D, exist_ok=True)

    def save(fig, name):
        fig.savefig(rf"{D}\{name}.pdf", bbox_inches="tight")
        fig.savefig(rf"{D}\{name}.png", dpi=600, bbox_inches="tight"); plt.close(fig)

    # ---------- train (resumes if interrupted) ----------
    torch.manual_seed(seed); np.random.seed(seed)
    m = timm.create_model(MODEL, pretrained=True, num_classes=len(LABELS)).cuda()
    o = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=1e-4)
    sc = torch.optim.lr_scheduler.CosineAnnealingLR(o, T_max=EPOCHS * len(tr))
    sl = torch.amp.GradScaler("cuda")
    hist, best, start, bad = [], 0.0, 0, 0

    if os.path.exists(CKPT):
        ck = torch.load(CKPT, map_location="cuda", weights_only=False)
        m.load_state_dict(ck["model"]); o.load_state_dict(ck["opt"])
        sc.load_state_dict(ck["sched"]); sl.load_state_dict(ck["scaler"])
        hist, best, start, bad = ck["hist"], ck["best"], ck["ep"] + 1, ck.get("bad", 0)
        if bad >= patience: start = EPOCHS
        print(f"seed {seed}: resumed, next epoch {start + 1}")

    for ep in range(start, EPOCHS):
        m.train(); t0 = time.time()
        rl = torch.zeros((), device="cuda"); ra = torch.zeros((), device="cuda"); n = 0
        for x, y in tqdm(tr, desc=f"seed {seed} epoch {ep+1}/{EPOCHS}"):
            x, y = x.cuda(non_blocking=True), y.cuda(non_blocking=True)
            o.zero_grad(set_to_none=True)
            with torch.autocast("cuda"):
                out = m(x); loss = lossf(out, y)
            sl.scale(loss).backward(); sl.step(o); sl.update(); sc.step()
            b = x.size(0); rl += loss.detach() * b
            ra += ((out.detach() > 0) == (y > 0.5)).float().mean() * b; n += b
        P, Y = _predict(m, va, "val"); v = evaluate(P, Y)
        row = dict(epoch=ep + 1, train_loss=float(rl) / n, train_acc=float(ra) / n,
                   val_loss=v["loss"], val_acc=v["acc"], val_f1=v["f1_macro"],
                   val_auroc=v["auroc"], sec=time.time() - t0)
        hist.append(row); pd.DataFrame(hist).to_csv(HIST, index=False)
        print(f"seed {seed} ep {ep+1:2d} | loss {row['train_loss']:.4f}/{row['val_loss']:.4f} | "
              f"val AUROC {row['val_auroc']:.4f} | {row['sec']/60:.1f} min")
        if v["auroc"] > best:
            best, bad = v["auroc"], 0; torch.save(m.state_dict(), BEST)
        else:
            bad += 1
        torch.save(dict(model=m.state_dict(), opt=o.state_dict(), sched=sc.state_dict(),
                        scaler=sl.state_dict(), hist=hist, best=best, ep=ep, bad=bad), CKPT)
        if bad >= patience:
            print(f"seed {seed}: early stop at epoch {ep+1}"); break

    # ---------- predict with the best epoch (cached on disk) ----------
    m.load_state_dict(torch.load(BEST, map_location="cuda", weights_only=True))
    SPL = {"train": ("Train", tr_eval), "val": ("Validation", va),
           "test": ("Test", te), "expert": ("Expert valid.", ex)}
    cache = rf"{D}\preds.npz"
    z = dict(np.load(cache)) if os.path.exists(cache) else {}
    for k, (name, dl) in SPL.items():
        if k == "train" and not eval_train: continue
        if k + "_P" not in z:
            z[k + "_P"], z[k + "_Y"] = _predict(m, dl, f"seed {seed} predict {name}")
    np.savez(cache, **z)
    preds = {name: (z[k + "_P"], z[k + "_Y"]) for k, (name, _) in SPL.items() if k + "_P" in z}

    # ---------- thresholds from VALIDATION only, then tables ----------
    Pv, Yv = preds["Validation"]
    THR_ = np.array([_best_thr(Pv[:, i], Yv[:, i].astype(int)) for i in range(len(LABELS))])
    rows, per = [], []
    for name, (P, Y) in preds.items():
        r = evaluate(P, Y, THR_)
        rows.append({"Split": name, "N": len(Y), "Accuracy": r["acc"], "Exact match": r["exact"],
                     "Macro-F1": r["f1_macro"], "Micro-F1": r["f1_micro"], "AUROC": r["auroc"]})
        pred, Yi = (P >= THR_).astype(int), Y.astype(int)
        for i, l in enumerate(LABELS):
            per.append({"Split": name, "Label": l, "Threshold": THR_[i], "Prevalence": Yi[:, i].mean(),
                        "Accuracy": (pred[:, i] == Yi[:, i]).mean(),
                        "Precision": precision_score(Yi[:, i], pred[:, i], zero_division=0),
                        "Recall": recall_score(Yi[:, i], pred[:, i], zero_division=0),
                        "F1": f1_score(Yi[:, i], pred[:, i], zero_division=0),
                        "AUROC": roc_auc_score(Yi[:, i], P[:, i])})
    summary, per = pd.DataFrame(rows), pd.DataFrame(per)
    summary.to_csv(rf"{D}\summary.csv", index=False); per.to_csv(rf"{D}\per_label.csv", index=False)
    np.save(rf"{D}\thresholds.npy", THR_)

    P, Y = preds["Test"]
    o2 = te_df[["img_path", "patient_id", "Sex", "Age"]].copy()
    for i, l in enumerate(LABELS):
        o2["y_" + l] = Y[:, i]; o2["p_" + l] = P[:, i]
    o2.to_csv(rf"{D}\test_preds.csv", index=False)

    # ---------- figures for this seed ----------
    h = pd.read_csv(HIST)
    fig, ax = plt.subplots(1, 3, figsize=(COL2, 2.2))
    ax[0].plot(h.epoch, h.train_loss, "-o", ms=2.5, label="Train"); ax[0].plot(h.epoch, h.val_loss, "--s", ms=2.5, label="Validation")
    ax[0].set(xlabel="Epoch", ylabel="BCE loss")
    ax[1].plot(h.epoch, h.train_acc, "-o", ms=2.5, label="Train"); ax[1].plot(h.epoch, h.val_acc, "--s", ms=2.5, label="Validation")
    ax[1].set(xlabel="Epoch", ylabel="Accuracy")
    ax[2].plot(h.epoch, h.val_f1, "-o", ms=2.5, label="Macro-F1"); ax[2].plot(h.epoch, h.val_auroc, "--s", ms=2.5, label="Mean AUROC")
    ax[2].set(xlabel="Epoch", ylabel="Validation score")
    for a in ax: a.grid(True, alpha=0.3); a.legend(frameon=False)
    fig.tight_layout(); save(fig, "fig_training_curves")

    styles = ["-", "--", "-.", ":", (0, (5, 1)), (0, (3, 1, 1, 1))]
    fig, ax = plt.subplots(figsize=(COL1, 3.0))
    for i, l in enumerate(LABELS):
        fpr, tpr, _ = roc_curve(Y[:, i], P[:, i])
        ax.plot(fpr, tpr, ls=styles[i], label=f"{l} ({roc_auc_score(Y[:, i], P[:, i]):.3f})")
    ax.plot([0, 1], [0, 1], "k:", lw=0.6)
    ax.set(xlabel="False positive rate", ylabel="True positive rate", xlim=(0, 1), ylim=(0, 1.02))
    ax.grid(True, alpha=0.3); ax.legend(loc="lower right", frameon=False)
    fig.tight_layout(); save(fig, "fig_roc_test")

    cms = multilabel_confusion_matrix(Y.astype(int), (P >= THR_).astype(int))
    fig, axes = plt.subplots(2, 3, figsize=(COL2, 4.3))
    for a, cm, l in zip(axes.ravel(), cms, LABELS):
        pct = cm / cm.sum(1, keepdims=True).clip(min=1)
        a.imshow(pct, cmap="Blues", vmin=0, vmax=1)
        for i in range(2):
            for j in range(2):
                a.text(j, i, f"{cm[i, j]:,}\n({pct[i, j]*100:.1f}%)", ha="center", va="center",
                       fontsize=6.5, color="white" if pct[i, j] > 0.5 else "black")
        a.set_xticks([0, 1]); a.set_xticklabels(["Neg", "Pos"]); a.set_yticks([0, 1]); a.set_yticklabels(["Neg", "Pos"])
        a.set(xlabel="Predicted", ylabel="Actual", title=l); a.grid(False)
    fig.tight_layout(); save(fig, "fig_cm_test")

    del m, o, sc, sl; gc.collect(); torch.cuda.empty_cache()
    return dict(summary=summary, per=per, THR=THR_)

print("run_seed ready, EPOCHS =", EPOCHS)

In [ ]:
RUN = [0, 1, 2]        # add 0 here to also train seed 0 with 7 epochs (recommended, see note below)
results = {}
for s in RUN:
    print(f"\n===== SEED {s} =====")
    results[s] = run_seed(s, patience=99, eval_train=True)
    print(results[s]["summary"].round(4).to_string(index=False))

In [ ]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve

MODEL, EPOCHS = "densenet121", 7
OUT = r"G:\runs"
LABELS = ["Cardiomegaly", "Edema", "Consolidation", "Atelectasis", "Pneumothorax", "Pleural Effusion"]
COL1, COL2 = 3.5, 7.16

plt.rcParams.update({
    "font.family": "serif", "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "stix", "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8,
    "xtick.labelsize": 7, "ytick.labelsize": 7, "legend.fontsize": 7,
    "axes.linewidth": 0.6, "lines.linewidth": 1.0, "grid.linewidth": 0.4,
    "figure.dpi": 150, "pdf.fonttype": 42, "ps.fonttype": 42,
})

# test-set table (needed for the fairness cell), same order as the predictions
df = pd.read_csv(r"G:\chexpert_final_clean.csv")
te_df = df[df.split == "test"].reset_index(drop=True)
print("test images:", len(te_df))     # expect 19074

In [ ]:
SAVE = rf"{OUT}\combined_e{EPOCHS}"; os.makedirs(SAVE, exist_ok=True)
SEEDS = [s for s in [0, 1, 2] if os.path.exists(rf"{OUT}\{MODEL}_e{EPOCHS}_seed{s}_results\preds.npz")]
print("seeds found:", SEEDS)

rd = lambda s, f: pd.read_csv(rf"{OUT}\{MODEL}_e{EPOCHS}_seed{s}_results\{f}").assign(seed=s)
summ = pd.concat([rd(s, "summary.csv") for s in SEEDS])
per = pd.concat([rd(s, "per_label.csv") for s in SEEDS])
Z = {s: np.load(rf"{OUT}\{MODEL}_e{EPOCHS}_seed{s}_results\preds.npz") for s in SEEDS}
pm = lambda x: f"{x.mean():.3f} ± {x.std():.3f}"

cols = ["Accuracy", "Exact match", "Macro-F1", "Micro-F1", "AUROC"]
overall = summ.groupby("Split", sort=False)[cols].agg(pm)
overall.insert(0, "N", summ.groupby("Split", sort=False)["N"].first())

pc = ["Accuracy", "Precision", "Recall", "F1", "AUROC"]
t = per[per.Split == "Test"]
perlab = t.groupby("Label", sort=False)[pc].agg(pm).loc[LABELS]
perlab.loc["Mean"] = t.groupby("seed")[pc].mean().agg(pm)

display(overall); display(perlab)
for name, tab in [("overall", overall), ("test_per_label", perlab)]:
    tab.to_csv(rf"{SAVE}\table_{name}.csv")
    open(rf"{SAVE}\table_{name}.tex", "w").write(tab.to_latex(escape=False).replace("±", r"$\pm$"))
print("saved to", SAVE)

In [ ]:
def sv(fig, name):
    fig.savefig(rf"{SAVE}\{name}.pdf", bbox_inches="tight")
    fig.savefig(rf"{SAVE}\{name}.png", dpi=600, bbox_inches="tight"); plt.show()

# 1. training curves, all seeds overlaid
H = {s: pd.read_csv(rf"{OUT}\{MODEL}_e{EPOCHS}_seed{s}_history.csv") for s in SEEDS}
fig, ax = plt.subplots(1, 3, figsize=(COL2, 2.2))
for s, h in H.items():
    c = f"C{s}"
    ax[0].plot(h.epoch, h.train_loss, "-", color=c, label=f"Train, seed {s}")
    ax[0].plot(h.epoch, h.val_loss, "--", color=c, label=f"Val, seed {s}")
    ax[1].plot(h.epoch, h.val_auroc, "-o", ms=2.5, color=c, label=f"Seed {s}")
    ax[2].plot(h.epoch, h.val_acc, "-o", ms=2.5, color=c, label=f"Seed {s}")
ax[0].set(xlabel="Epoch", ylabel="BCE loss"); ax[1].set(xlabel="Epoch", ylabel="Validation mean AUROC")
ax[2].set(xlabel="Epoch", ylabel="Validation accuracy")
for a in ax: a.grid(True, alpha=0.3)
ax[0].legend(frameon=False, fontsize=5.5); ax[1].legend(frameon=False)
fig.tight_layout(); sv(fig, "fig_curves_all_seeds")

# 2. per-label test AUROC, mean ± std over seeds
pt = per[per.Split == "Test"].pivot(index="Label", columns="seed", values="AUROC").loc[LABELS]
fig, ax = plt.subplots(figsize=(COL1, 2.6))
ax.bar(range(len(LABELS)), pt.mean(1), yerr=pt.std(1), capsize=2, color="0.6", edgecolor="k", lw=0.5)
ax.set_xticks(range(len(LABELS))); ax.set_xticklabels(LABELS, rotation=40, ha="right")
ax.set(ylabel="Test AUROC", ylim=(0.6, 0.95)); ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout(); sv(fig, "fig_auroc_per_label")

# 3. ROC per disease, one curve per seed
styles = ["-", "--", "-."]
fig, axes = plt.subplots(2, 3, figsize=(COL2, 4.2))
for i, (a, l) in enumerate(zip(axes.ravel(), LABELS)):
    for s in SEEDS:
        P_, Y_ = Z[s]["test_P"], Z[s]["test_Y"]
        fpr, tpr, _ = roc_curve(Y_[:, i], P_[:, i])
        a.plot(fpr, tpr, ls=styles[s], label=f"Seed {s} ({roc_auc_score(Y_[:, i], P_[:, i]):.3f})")
    a.plot([0, 1], [0, 1], "k:", lw=0.6); a.set_title(l)
    a.set(xlabel="False positive rate", ylabel="True positive rate"); a.grid(True, alpha=0.3)
    a.legend(loc="lower right", frameon=False, fontsize=6)
fig.tight_layout(); sv(fig, "fig_roc_all_seeds")

In [ ]:
grp = {"Male": te_df.Sex.eq("Male"), "Female": te_df.Sex.eq("Female"),
       "Age <50": te_df.Age < 50, "Age 50-65": te_df.Age.between(50, 65), "Age >65": te_df.Age > 65}
rows = []
for s in SEEDS:
    P_, Y_ = Z[s]["test_P"], Z[s]["test_Y"]
    for g, msk in grp.items():
        mm = msk.values
        rows.append({"seed": s, "Group": g, "N": int(mm.sum()),
                     "Mean AUROC": np.mean([roc_auc_score(Y_[mm, i], P_[mm, i]) for i in range(len(LABELS))])})
fr = pd.DataFrame(rows)
fair = fr.groupby("Group", sort=False).agg(N=("N", "first"), AUROC=("Mean AUROC", pm))
display(fair)
fair.to_csv(rf"{SAVE}\table_fairness.csv")
open(rf"{SAVE}\table_fairness.tex", "w").write(fair.to_latex(escape=False).replace("±", r"$\pm$"))

Pe = np.mean([Z[s]["test_P"] for s in SEEDS], axis=0); Ye = Z[SEEDS[0]]["test_Y"]
ens = [roc_auc_score(Ye[:, i], Pe[:, i]) for i in range(len(LABELS))]
single = per[per.Split == "Test"].groupby("seed")["AUROC"].mean()
print(f"single-seed mean AUROC: {single.mean():.4f} ± {single.std():.4f}")
print(f"{len(SEEDS)}-seed ensemble AUROC: {np.mean(ens):.4f}")

In [ ]:
rng = np.random.default_rng(0)
B = 500

pid = te_df.patient_id.values
plist = pd.Series(np.arange(len(te_df))).groupby(pid).apply(np.array).values
Ys = Z[SEEDS[0]]["test_Y"]
Ps = {s: Z[s]["test_P"] for s in SEEDS}
Pe = np.mean(list(Ps.values()), axis=0)                  # ensemble

def lab_aucs(P, idx):
    return np.array([roc_auc_score(Ys[idx, i], P[idx, i]) for i in range(len(LABELS))])

full = np.arange(len(te_df))
pt_single = np.mean([lab_aucs(Ps[s], full) for s in SEEDS], axis=0)    # seed-averaged, all test data
pt_ens = lab_aucs(Pe, full)

bs_single, bs_ens = [], []
for b in range(B):
    idx = np.concatenate(plist[rng.integers(0, len(plist), len(plist))])   # same patients for every seed
    bs_single.append(np.mean([lab_aucs(Ps[s], idx) for s in SEEDS], axis=0))
    bs_ens.append(lab_aucs(Pe, idx))
bs_single, bs_ens = np.array(bs_single), np.array(bs_ens)

def ci(a, axis=0):
    return np.percentile(a, [2.5, 97.5], axis=axis)

rows = []
for i, l in enumerate(LABELS):
    lo, hi = ci(bs_single[:, i])
    rows.append({"Label": l, "AUROC": pt_single[i], "CI low": lo, "CI high": hi,
                 "Seed std": per[(per.Split == "Test") & (per.Label == l)].AUROC.std()})
m_boot = bs_single.mean(1)
lo, hi = ci(m_boot)
rows.append({"Label": "Mean", "AUROC": pt_single.mean(), "CI low": lo, "CI high": hi,
             "Seed std": per[per.Split == "Test"].groupby("seed").AUROC.mean().std()})
tab = pd.DataFrame(rows)
tab["95% CI"] = tab.apply(lambda r: f"[{r['CI low']:.3f}, {r['CI high']:.3f}]", axis=1)
display(tab[["Label", "AUROC", "95% CI", "Seed std"]].round(4))

lo, hi = ci(bs_ens.mean(1))
print(f"Ensemble mean AUROC: {pt_ens.mean():.4f}  95% CI [{lo:.4f}, {hi:.4f}]")

# age and sex gaps, seed-averaged, bootstrapped by patient
sex, age = te_df.Sex.values, te_df.Age.values
mean_auc = lambda P, idx: lab_aucs(P, idx).mean()
d_sex, d_age = [], []
for b in range(B):
    idx = np.concatenate(plist[rng.integers(0, len(plist), len(plist))])
    s_, a_ = sex[idx], age[idx]
    d_sex.append(np.mean([mean_auc(Ps[s], idx[s_ == "Female"]) - mean_auc(Ps[s], idx[s_ == "Male"]) for s in SEEDS]))
    d_age.append(np.mean([mean_auc(Ps[s], idx[a_ < 50]) - mean_auc(Ps[s], idx[a_ > 65]) for s in SEEDS]))
for name, d in [("Female - Male", d_sex), ("Age<50 - Age>65", d_age)]:
    lo, hi = np.percentile(d, [2.5, 97.5])
    print(f"{name}: {np.mean(d):+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]")

tab.drop(columns="95% CI").to_csv(rf"{SAVE}\table_bootstrap_ci.csv", index=False)
print("saved to", SAVE)